# Dating RL v1: Tabular Q-Learning From Scratch

This notebook builds a small reinforcement-learning experiment using only standard Python plus NumPy. There is no Gymnasium, no PPO, no deep RL, no virtual environment, no database, and no API.

The goal is to compare three policies in a toy dating/conversation simulator:

- random actions
- hand-written rule-based actions
- learned tabular Q-learning actions

The main metric is `date_rate`: how often an episode ends in `date_success`.


## 1. Imports and Random Seeds

We only use `random` from standard Python and `numpy` for arrays, random sampling, and Q-table math.

In [ ]:
import random
import time

import numpy as np

np.random.seed(7)
random.seed(7)

%pip install openai python-dotenv


## 2. State Space and Action Space

The state has 3 parts:

- `interest`: low, medium, high
- `stage`: opener, chat, date
- `tone`: cold, neutral, warm

That gives `3 x 3 x 3 = 27` discrete states.

The agent has 8 possible actions.

In [ ]:
INTERESTS = ["low", "medium", "high"]
STAGES = ["opener", "chat", "date"]
TONES = ["cold", "neutral", "warm"]

ACTIONS = [
    "ask_question",
    "give_compliment",
    "share_story",
    "be_playful",
    "be_direct",
    "suggest_date",
    "slow_down",
    "end_chat",
]

N_INTEREST = len(INTERESTS)
N_STAGE = len(STAGES)
N_TONE = len(TONES)
N_STATES = N_INTEREST * N_STAGE * N_TONE
N_ACTIONS = len(ACTIONS)

print(f"Number of states: {N_STATES}")
print(f"Number of actions: {N_ACTIONS}")

## 3. State Encoding Helpers

A Q-table needs integer state IDs. These helpers convert between a readable state tuple and a number from `0` to `26`.

In [ ]:
def state_to_id(interest, stage, tone):
    """Convert a state tuple into one integer from 0 to 26."""
    return interest * (N_STAGE * N_TONE) + stage * N_TONE + tone


def id_to_state(state_id):
    """Convert one integer from 0 to 26 back into a state tuple."""
    interest = state_id // (N_STAGE * N_TONE)
    rest = state_id % (N_STAGE * N_TONE)
    stage = rest // N_TONE
    tone = rest % N_TONE
    return interest, stage, tone


def describe_state(state_id):
    interest, stage, tone = id_to_state(state_id)
    return f"interest={INTERESTS[interest]:6s} stage={STAGES[stage]:6s} tone={TONES[tone]}"


def clamp(value, low, high):
    return max(low, min(high, value))


for state_id in range(5):
    print(state_id, "->", describe_state(state_id))

## 4. Episode Reset

Each episode starts at the opener stage. Interest starts as low or medium, and tone starts cold or neutral.

In [ ]:
def reset_episode():
    """Standard starting distribution used for training and fair evaluation."""
    interest = np.random.choice([0, 1], p=[0.55, 0.45])
    stage = 0
    tone = np.random.choice([0, 1], p=[0.35, 0.65])
    return state_to_id(interest, stage, tone)


def reset_episode_hard():
    """Harder starting distribution with more cold/low-interest openers."""
    interest = np.random.choice([0, 1], p=[0.60, 0.40])
    stage = 0
    tone = np.random.choice([0, 1], p=[0.60, 0.40])
    return state_to_id(interest, stage, tone)


start_state = reset_episode()
print(start_state, describe_state(start_state))


## 5. Hand-Coded Stochastic Simulator

This is the toy environment. It takes the current state and one action, then returns:

- the next state
- the reward
- whether the episode is done

The rules are intentionally simple and editable. Good timing tends to increase interest and tone. Pushy timing can lower both.

In [ ]:
def compute_reward(old_state, new_state, outcome):
    """Reward depends on state progress and terminal outcomes, not directly on action names."""
    if outcome == "date_success":
        return 30.0
    if outcome == "date_failed":
        return -5.0
    if outcome == "ghosted":
        return -5.0
    if outcome == "ended_by_agent":
        return -5.0   # changed from -1.0; quitting is now genuinely costly

    old_i, old_s, old_t = old_state
    new_i, new_s, new_t = new_state
    progress_reward = (
        0.25 * (new_s - old_s)
        + 0.15 * (new_i - old_i)
        + 0.05 * (new_t - old_t)
    )
    time_cost = -0.10
    return progress_reward + time_cost


def simulator_step(state_id, action):
    """Hand-coded stochastic simulator returning next_state, reward, done, outcome."""
    interest, stage, tone = id_to_state(state_id)
    old_state = (interest, stage, tone)
    done = False
    outcome = None
    noise = np.random.choice([-1, 0, 1], p=[0.10, 0.80, 0.10])

    if action == 0:  # ask_question
        interest += np.random.choice([0, 1], p=[0.55, 0.45])
        tone += np.random.choice([0, 1], p=[0.50, 0.50])

    elif action == 1:  # give_compliment
        if tone >= 1:
            interest += 1
        else:
            tone -= 1

    elif action == 2:  # share_story
        if stage >= 1:
            interest += np.random.choice([0, 1], p=[0.45, 0.55])
            tone += 1

    elif action == 3:  # be_playful
        if tone == 2:
            interest += 1
        else:
            tone += np.random.choice([-1, 1], p=[0.45, 0.55])

    elif action == 4:  # be_direct
        if interest >= 1 and tone >= 1:
            stage += 1
        else:
            interest -= 1
            tone -= 1

    elif action == 5:  # suggest_date
        done = True
        if interest == 2 and tone == 2:
            outcome = "date_success" if np.random.random() < 0.75 else "date_failed"
        elif interest >= 1 and stage == 2:
            outcome = "date_success" if np.random.random() < 0.45 else "date_failed"
        else:
            outcome = "date_failed"

    elif action == 6:  # slow_down
        tone += 1
        if interest == 0 and np.random.random() < 0.30:
            interest += 1

    elif action == 7:  # end_chat
        done = True
        outcome = "ended_by_agent"

    interest = clamp(interest + noise, 0, N_INTEREST - 1)
    stage = clamp(stage, 0, N_STAGE - 1)
    tone = clamp(tone, 0, N_TONE - 1)
    next_state_id = state_to_id(interest, stage, tone)
    new_state = (interest, stage, tone)

    if not done and np.random.random() < 0.03:
        done = True
        outcome = "ghosted"

    reward = compute_reward(old_state, new_state, outcome)
    return next_state_id, reward, done, outcome


## 6. Try One Simulator Step

This small test lets us see how one action changes the state. The simulator now also returns an `outcome`, which stays `None` unless the episode ended.


In [ ]:
state = reset_episode()
action = ACTIONS.index("ask_question")
next_state, reward, done, outcome = simulator_step(state, action)

print("Before:", describe_state(state))
print("Action:", ACTIONS[action])
print("After: ", describe_state(next_state))
print("Reward:", round(reward, 3), "Done:", done, "Outcome:", outcome)


## 7. Epsilon-Greedy Action Selection

The agent mostly chooses the best known action, but sometimes explores random actions.

- high epsilon = more exploration
- low epsilon = more exploitation

In [ ]:
def choose_action(Q, state_id, epsilon):
    """Epsilon-greedy action choice."""
    if np.random.random() < epsilon:
        return np.random.randint(N_ACTIONS)
    return int(np.argmax(Q[state_id]))

## 8. Rule-Based Policy

This is a hand-coded common-sense baseline. It does not learn. It gives the Q-learner a stronger comparison than random actions.

The rules are intentionally reasonable:

- suggest a date when interest and tone are both high
- avoid playful/direct moves when tone is cold
- warm up low-interest states
- ask questions early
- build rapport in the chat stage


In [ ]:
def rule_based_action(state_id):
    """A simple common-sense policy with no learning."""
    interest, stage, tone = id_to_state(state_id)

    if interest == 2 and tone == 2:
        return ACTIONS.index("suggest_date")
    if stage == 2 and interest >= 1 and tone >= 1:
        return ACTIONS.index("suggest_date")
    if tone == 0:
        return ACTIONS.index("ask_question")
    if interest == 0:
        return ACTIONS.index("slow_down")
    if stage == 0:
        return ACTIONS.index("ask_question")
    if stage == 1 and tone == 2:
        return ACTIONS.index("be_playful")
    if stage == 1:
        return ACTIONS.index("share_story")
    return ACTIONS.index("be_direct")


for state_id in range(0, N_STATES, 5):
    action = rule_based_action(state_id)
    print(f"{describe_state(state_id)} -> {ACTIONS[action]}")


## 9. Run One Episode

Episodes are variable length. They can end because the agent suggests a date, ends the chat, the conversation fizzles out, or the maximum step limit is reached.

When `learn=True`, this function also updates the Q-table. When `policy_fn` is provided, it uses that hand-coded policy instead of random or Q-table actions.


In [ ]:
def run_episode(
    Q=None,
    policy_fn=None,
    epsilon=0.0,
    max_steps=25,
    reset_fn=reset_episode,
    learn=False,
    alpha=0.1,
    gamma=0.95,
):
    """Run one variable-length episode. If learn=True, update Q in place."""
    state_id = reset_fn()
    total_reward = 0.0

    for step in range(max_steps):
        if policy_fn is not None:
            action = policy_fn(state_id)
        elif Q is None:
            action = np.random.randint(N_ACTIONS)
        else:
            action = choose_action(Q, state_id, epsilon)

        next_state_id, reward, done, outcome = simulator_step(state_id, action)
        total_reward += reward

        if learn:
            best_next = np.max(Q[next_state_id])
            target = reward + (0.0 if done else gamma * best_next)
            Q[state_id, action] += alpha * (target - Q[state_id, action])

        state_id = next_state_id
        if done:
            return total_reward, step + 1, outcome

    return total_reward, max_steps, "max_steps"


reward, length, outcome = run_episode(Q=None)
print(f"One random episode: reward={reward:.3f}, length={length}, outcome={outcome}")


## 10. Random and Rule-Based Baselines

Before training, we measure two baselines:

- `random`: chooses random actions
- `rule-based`: uses our hand-written common-sense rules

Both return average reward, average episode length, and date success rate.


In [ ]:
def summarize_policy(episodes=1000, Q=None, policy_fn=None, reset_fn=reset_episode):
    rewards = []
    lengths = []
    n_dates = 0
    for _ in range(episodes):
        reward, length, outcome = run_episode(Q=Q, policy_fn=policy_fn, reset_fn=reset_fn)
        rewards.append(reward)
        lengths.append(length)
        if outcome == "date_success":
            n_dates += 1
    return np.mean(rewards), np.mean(lengths), n_dates / episodes


def random_baseline(episodes=1000, reset_fn=reset_episode):
    return summarize_policy(episodes, reset_fn=reset_fn)


def rule_based_baseline(episodes=1000, reset_fn=reset_episode):
    return summarize_policy(episodes, policy_fn=rule_based_action, reset_fn=reset_fn)


base_reward, base_length, base_date_rate = random_baseline()
rule_reward, rule_length, rule_date_rate = rule_based_baseline()

print(f"Random:     avg_reward={base_reward:.3f}  length={base_length:.2f}  date_rate={base_date_rate:.1%}")
print(f"Rule-based: avg_reward={rule_reward:.3f}  length={rule_length:.2f}  date_rate={rule_date_rate:.1%}")


## 11. Train Tabular Q-Learning

The Q-table has one row per state and one column per action.

Q-learning update:

`Q[state, action] = Q[state, action] + alpha * (target - Q[state, action])`

where the target is the reward plus the discounted value of the best next action.


In [ ]:
def train_q_learning(episodes=8000, reset_fn=reset_episode):
    Q = np.zeros((N_STATES, N_ACTIONS))
    rewards = []

    for episode in range(episodes):
        epsilon = max(0.05, 1.0 - episode / (episodes * 0.75))
        reward, _, _ = run_episode(Q=Q, epsilon=epsilon, reset_fn=reset_fn, learn=True)
        rewards.append(reward)

    return Q, rewards


Q, training_rewards = train_q_learning()
print("Q-table shape:", Q.shape)
print(f"Last 500 episode avg reward: {np.mean(training_rewards[-500:]):.3f}")


## 12. Evaluate the Learned Policy

Now we run episodes with `epsilon=0`, meaning the agent always chooses the best learned action.

We compare the learned policy against both baselines using the headline metric: `date_rate`.


In [ ]:
def evaluate_any_policy(name, episodes=1000, Q=None, policy_fn=None, reset_fn=reset_episode, sarsa=False):
    rewards = []
    lengths = []
    n_dates = 0
    for _ in range(episodes):
        if sarsa:
            reward, length, outcome = run_episode_sarsa(Q, epsilon=0.0, reset_fn=reset_fn)
        else:
            reward, length, outcome = run_episode(Q=Q, policy_fn=policy_fn, reset_fn=reset_fn)
        rewards.append(reward)
        lengths.append(length)
        if outcome == "date_success":
            n_dates += 1
    return {
        "policy": name,
        "avg_reward": float(np.mean(rewards)),
        "avg_length": float(np.mean(lengths)),
        "date_rate": n_dates / episodes,
    }


def evaluate_policy(Q, episodes=1000, reset_fn=reset_episode):
    result = evaluate_any_policy("Q-learning", episodes=episodes, Q=Q, reset_fn=reset_fn)
    return result["avg_reward"], result["avg_length"], result["date_rate"]


learned_reward, learned_length, learned_date_rate = evaluate_policy(Q)

print(f"Random:     avg_reward={base_reward:.3f}  length={base_length:.2f}  date_rate={base_date_rate:.1%}")
print(f"Rule-based: avg_reward={rule_reward:.3f}  length={rule_length:.2f}  date_rate={rule_date_rate:.1%}")
print(f"Q-learned:  avg_reward={learned_reward:.3f}  length={learned_length:.2f}  date_rate={learned_date_rate:.1%}")


## 13. Print the Learned Policy

For each of the 27 states, this prints the action with the highest Q-value.


In [ ]:
def print_learned_policy(Q):
    print("\nLearned policy:")
    print("-" * 78)
    for state_id in range(N_STATES):
        best_action = int(np.argmax(Q[state_id]))
        value = Q[state_id, best_action]
        print(f"{state_id:02d} | {describe_state(state_id)} -> {ACTIONS[best_action]:14s} Q={value:6.2f}")


print_learned_policy(Q)

## 17. Train Tabular SARSA

SARSA is another tabular reinforcement-learning algorithm. Q-learning is off-policy because it learns from the best possible next action. SARSA is on-policy because it learns from the next action it actually chooses under epsilon-greedy exploration.


In [ ]:
def run_episode_sarsa(
    Q,
    epsilon=0.0,
    max_steps=25,
    reset_fn=reset_episode,
    learn=False,
    alpha=0.1,
    gamma=0.95,
):
    state_id = reset_fn()
    action = choose_action(Q, state_id, epsilon)
    total_reward = 0.0

    for step in range(max_steps):
        next_state_id, reward, done, outcome = simulator_step(state_id, action)
        total_reward += reward

        if done:
            if learn:
                Q[state_id, action] += alpha * (reward - Q[state_id, action])
            return total_reward, step + 1, outcome

        next_action = choose_action(Q, next_state_id, epsilon)
        if learn:
            target = reward + gamma * Q[next_state_id, next_action]
            Q[state_id, action] += alpha * (target - Q[state_id, action])

        state_id = next_state_id
        action = next_action

    return total_reward, max_steps, "max_steps"


def train_sarsa(episodes=8000, reset_fn=reset_episode):
    Q_sarsa = np.zeros((N_STATES, N_ACTIONS))
    rewards = []

    for episode in range(episodes):
        epsilon = max(0.05, 1.0 - episode / (episodes * 0.75))
        reward, _, _ = run_episode_sarsa(
            Q=Q_sarsa,
            epsilon=epsilon,
            reset_fn=reset_fn,
            learn=True,
            alpha=0.1,
            gamma=0.95,
        )
        rewards.append(reward)

    return Q_sarsa, rewards


Q_sarsa, sarsa_training_rewards = train_sarsa()
print("SARSA Q-table shape:", Q_sarsa.shape)
print(f"Last 500 episode avg reward: {np.mean(sarsa_training_rewards[-500:]):.3f}")


## 18. Fair Policy Comparison

This evaluates all four policies with the same simulator, same episode count, and same reset distribution. Use `reset_episode` for the standard test or `reset_episode_hard` for a harder test.


In [ ]:
def compare_policies(Q, Q_sarsa, episodes=3000, reset_fn=reset_episode):
    results = [
        evaluate_any_policy("Random", episodes=episodes, reset_fn=reset_fn),
        evaluate_any_policy("Rule-based", episodes=episodes, policy_fn=rule_based_action, reset_fn=reset_fn),
        evaluate_any_policy("Q-learning", episodes=episodes, Q=Q, reset_fn=reset_fn),
        evaluate_any_policy("SARSA", episodes=episodes, Q=Q_sarsa, reset_fn=reset_fn, sarsa=True),
    ]

    print(f"{'Policy':<14} {'avg_reward':>10} {'length':>8} {'date_rate':>10}")
    print("-" * 47)
    for row in results:
        print(
            f"{row['policy']:<14} {row['avg_reward']:>10.3f} "
            f"{row['avg_length']:>8.2f} {row['date_rate']:>9.1%}"
        )
    return results


standard_results = compare_policies(Q, Q_sarsa, episodes=3000, reset_fn=reset_episode)
hard_results = compare_policies(Q, Q_sarsa, episodes=3000, reset_fn=reset_episode_hard)


## 19. Print SARSA Learned Policy

This prints the greedy action selected by SARSA for each of the 27 states.


In [ ]:
print_learned_policy(Q_sarsa)


## 14. What to Edit Next

Good beginner experiments:

- change the rewards in `simulator_step`
- change the rule-based policy
- change the action list
- change `max_steps` in `run_episode`
- train for more or fewer episodes
- inspect the Q-table directly with `Q`


## 15. Kimi LLM State Classifier

This section converts a real dating-app message from the other person into the same `(interest, stage, tone)` state format used by the Q-table.

It uses Kimi through the OpenAI-compatible Python client. The notebook expects these environment variables to already be loaded earlier:

- `KIMI_API_KEY`
- `KIMI_BASE_URL`
- `KIMI_MODEL`

The classifier returns a tuple of three integers, for example `(2, 1, 2)` for high interest, chat stage, warm tone.


### 15.1 Label Mappings

The LLM returns string labels, but the Q-table uses integer state IDs. These dictionaries translate labels into the same integer indices used by `INTERESTS`, `STAGES`, and `TONES`.


In [ ]:
INTEREST_MAP = {"low": 0, "medium": 1, "high": 2}
STAGE_MAP = {"opener": 0, "chat": 1, "date": 2}
TONE_MAP = {"cold": 0, "neutral": 1, "warm": 2}


### 15.2 Classifier System Prompt

The prompt defines each label clearly so the model uses the same meaning as the simulator.


In [ ]:
CLASSIFIER_SYSTEM_PROMPT = """You are a precise classifier of dating-app conversation states.

You will be given a single message from one person. Optionally, you may be given prior conversation context.

Your job is to classify the message into three fields:

* "interest": one of ["low", "medium", "high"]
* "stage": one of ["opener", "chat", "date"]
* "tone": one of ["cold", "neutral", "warm"]

Return ONLY valid JSON in this format:
{"interest":"...", "stage":"...", "tone":"..."}

---

DEFINITIONS:

INTEREST (engagement level):

* low: very short, dismissive, dry replies; no curiosity; no follow-up
* medium: cooperative but not enthusiastic; polite replies like "sure", "okay"
* high: enthusiastic, expressive, asks questions, shows clear interest

STAGE (conversation progression):

* opener: first message or simple greeting (e.g., "hey", "hi", "hello")
* chat: normal back-and-forth conversation
* date: discussing meeting in person, plans, or clear escalation

TONE (emotional style):

* cold: flat, dismissive, reluctant, low effort
* neutral: polite but not expressive
* warm: friendly, playful, enthusiastic, expressive

---

CRITICAL RULES:

1. SHORT REPLIES:

* "k", "ok", "idk", "maybe", "idk maybe", "i guess" → interest = low, tone = cold
* "sure" → interest = medium, tone = neutral

2. STAGE DISAMBIGUATION:

* Only greetings like "hey", "hi", "hello" → stage = opener
* If the message looks like a reply (e.g., "idk maybe", "i guess", "sure", "lol"), DO NOT classify as opener → use "chat"
* If no context is given, default to "chat" unless it is clearly a greeting

3. HIGH INTEREST SIGNALS:

* enthusiasm, exclamation marks, caps ("OMG", "YES")
* laughter + engagement ("haha that's funny")
* asking questions back
  → interest = high, tone = warm

4. DATE STAGE:

* any message suggesting meeting ("drink", "this weekend", "hang out", "grab coffee")
  → stage = date

5. TONE PRIORITY:

* dismissive/reluctant → cold
* polite/simple → neutral
* expressive/playful → warm

---

IMPORTANT:

* Be strict and consistent.
* Do not overestimate interest.
* When uncertain, prefer lower interest and "chat" stage.

---

OUTPUT:
Return ONLY JSON.
No explanation. No extra text.
"""

### 15.3 Kimi Client Setup

This creates an OpenAI-compatible client pointed at Kimi. It raises a clear error if the required environment variables are missing.


In [ ]:
import json
import os

from openai import OpenAI

required_vars = ["KIMI_API_KEY", "KIMI_BASE_URL", "KIMI_MODEL"]
missing_vars = [name for name in required_vars if not os.getenv(name)]
if missing_vars:
    raise ValueError(f"Missing environment variable(s): {', '.join(missing_vars)}")

client = OpenAI(
    api_key=os.getenv("KIMI_API_KEY"),
    base_url=os.getenv("KIMI_BASE_URL"),
)
KIMI_MODEL = os.getenv("KIMI_MODEL")


### 15.4 `classify(text, history=None)`

This function sends the latest message to Kimi, asks for JSON only, validates the returned labels, and converts the result into a tuple of three integers.


In [ ]:
def classify(text, history=None):
    """
    Classify one message into an (interest, stage, tone) integer state.

    text: most recent message from the OTHER person
    history: optional list of prior messages for context

    Returns: tuple of three ints
    Raises: ValueError if the LLM returns missing or invalid labels
    """
    if history:
        history_text = "\n".join(f"- {m}" for m in history)
        user_msg = f"Conversation history:\n{history_text}\n\nLatest message to classify:\n{text}"
    else:
        user_msg = f"Message to classify:\n{text}"

    response = client.chat.completions.create(
        model=KIMI_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content
    data = json.loads(raw)

    expected_keys = {"interest", "stage", "tone"}
    if set(data) != expected_keys:
        raise ValueError(
            f"Expected exactly {sorted(expected_keys)}, got {sorted(data)}. Raw output: {raw!r}"
        )

    if data["interest"] not in INTEREST_MAP:
        raise ValueError(f"Invalid interest label: {data['interest']!r}. Raw output: {raw!r}")
    if data["stage"] not in STAGE_MAP:
        raise ValueError(f"Invalid stage label: {data['stage']!r}. Raw output: {raw!r}")
    if data["tone"] not in TONE_MAP:
        raise ValueError(f"Invalid tone label: {data['tone']!r}. Raw output: {raw!r}")

    return (
        INTEREST_MAP[data["interest"]],
        STAGE_MAP[data["stage"]],
        TONE_MAP[data["tone"]],
    )


### 15.5 Hand-Labeled Validation Examples

Run this cell after your Kimi environment variables are loaded. It compares the classifier against hand-labeled examples so you can quickly see whether its labels match your intuition.


In [ ]:
# Hand-labeled classifier validation examples.
test_cases = [
    ("hey", "low", "opener", "neutral"),
    ("lol that's actually really cute haha", "high", "chat", "warm"),
    ("k", "low", "opener", "cold"),
    ("yeah I love hiking too! where do you usually go?", "high", "chat", "warm"),
    ("wanna grab a drink this weekend?", "high", "date", "warm"),
    ("idk maybe", "low", "chat", "cold"),
    ("haha you're funny, what do you do for work?", "high", "chat", "warm"),
    ("i guess", "low", "chat", "cold"),
    ("OMG YES I LOVE THAT BAND", "high", "chat", "warm"),
    ("sure", "medium", "chat", "neutral"),
]

print(f"{'Message':<55} {'Expected':<25} {'Got':<25} {'Match'}")
print("-" * 115)

n_correct = 0
for msg, exp_i, exp_s, exp_t in test_cases:
    got_i, got_s, got_t = classify(msg)
    expected = f"({exp_i}, {exp_s}, {exp_t})"
    got = f"({INTERESTS[got_i]}, {STAGES[got_s]}, {TONES[got_t]})"
    match = (
        INTERESTS[got_i] == exp_i and
        STAGES[got_s] == exp_s and
        TONES[got_t] == exp_t
    )
    if match:
        n_correct += 1
    msg_display = msg if len(msg) <= 50 else msg[:47] + "..."
    print(f"{msg_display:<55} {expected:<25} {got:<25} {'yes' if match else 'no'}")

print(f"\nAccuracy: {n_correct}/{len(test_cases)} = {n_correct / len(test_cases):.0%}")


In [ ]:
# Query Moonshot to list models YOUR account has access to
response = client.models.list()
for m in response.data:
    print(m.id)
    

## 16. Interactive Agent Interface

This cell turns the trained Q-table into a small demo.

It takes one message from the other person, classifies it into `(interest, stage, tone)`, converts that state into the Q-table state ID, chooses the best learned action, and prints a readable next-move suggestion.

For now, this does **not** generate the final text reply. It only shows:

`message -> state -> action -> suggested next move`


In [ ]:
ACTION_GUIDANCE = {
    "ask_question": "Ask an open-ended question to keep the conversation moving.",
    "give_compliment": "Give a specific, light compliment without overdoing it.",
    "share_story": "Share a short personal story that builds rapport.",
    "be_playful": "Use playful banter or gentle teasing.",
    "be_direct": "Use direct but warm escalation.",
    "suggest_date": "Suggest a simple, low-pressure plan to meet.",
    "slow_down": "Give them space and respond calmly without pushing.",
    "end_chat": "End the chat politely instead of forcing the conversation.",
}


def agent_interface(message, history=None, Q_table=None, label="Q-learning"):
    """
    Classify a real message, choose the best learned action,
    and print the agent's recommended strategy.
    """
    if Q_table is None:
        if "Q" not in globals():
            raise ValueError("Q-table not found. Run the training cell first so Q exists.")
        Q_table = Q

    interest, stage, tone = classify(message, history=history)
    state_id = state_to_id(interest, stage, tone)
    action_id = int(np.argmax(Q_table[state_id]))
    action_name = ACTIONS[action_id]

    print(f"Input: {message}")
    print("Classified state:")
    print(f"  interest={INTERESTS[interest]}, stage={STAGES[stage]}, tone={TONES[tone]}")
    print(f"Agent strategy ({label}): {action_name}")
    print(f"Suggested next move: {ACTION_GUIDANCE[action_name]}")

    return {
        "message": message,
        "state": (interest, stage, tone),
        "state_id": state_id,
        "action_id": action_id,
        "action": action_name,
        "suggested_next_move": ACTION_GUIDANCE[action_name],
    }


# Example demo. This calls classify(), so run it only after the Kimi client cell is configured.
demo = agent_interface("haha you're funny, what do you do for work?")


## 20. SARSA Agent Interface

This uses the same classifier and display format, but chooses actions from the SARSA Q-table.


In [ ]:
def agent_interface_sarsa(message, history=None):
    if "Q_sarsa" not in globals():
        raise ValueError("SARSA Q-table not found. Run the SARSA training cell first.")
    return agent_interface(message, history=history, Q_table=Q_sarsa, label="SARSA")


# Example demo. This calls classify(), so run it only after the Kimi client cell is configured.
demo_sarsa = agent_interface_sarsa("haha you're funny, what do you do for work?")


## 21. LLM World-Model Simulator

This is an additive simulator that uses Kimi as the world model. It does **not** replace `simulator_step`; it gives you a separate function for experiments where the other person's response and next state come from an LLM instead of the hand-coded transition rules.

Return format:

`next_state_id, reward, done, outcome, response_text`

Use this carefully because each call spends API credits.


In [ ]:
LLM_OUTCOME_VALUES = {None, "date_success", "date_failed", "ghosted"}


def _format_history_for_llm(history):
    if not history:
        return "No prior messages. This is the first turn."

    lines = []
    for turn_number, turn in enumerate(history, start=1):
        agent_msg = turn.get("agent")
        other_msg = turn.get("other")
        lines.append(f"Turn {turn_number}:")
        if agent_msg:
            lines.append(f"  Agent: {agent_msg}")
        if other_msg:
            lines.append(f"  Other person: {other_msg}")
    return "\n".join(lines)


def simulator_step_llm(state_id, action, history):
    """
    LLM-backed world-model step.

    Inputs:
        state_id: integer 0-26
        action: integer 0-7
        history: list of {"agent": "...", "other": "..."} dicts

    Returns:
        next_state_id, reward, done, outcome, response_text
    """
    old_state = id_to_state(state_id)
    old_interest, old_stage, old_tone = old_state
    action_name = ACTIONS[action]

    if action_name == "end_chat":
        outcome = "ended_by_agent"
        reward = compute_reward(old_state, old_state, outcome)
        return state_id, reward, True, outcome, ""

    current_state_text = (
        f"interest={INTERESTS[old_interest]}, "
        f"stage={STAGES[old_stage]}, "
        f"tone={TONES[old_tone]}"
    )
    action_description = ACTION_GUIDANCE.get(action_name, "Use the named strategy naturally.")
    history_text = _format_history_for_llm(history)

    system_prompt = f"""You are roleplaying as the OTHER person in a dating-app conversation.

Current emotional conversation state:
- interest: {INTERESTS[old_interest]}
- stage: {STAGES[old_stage]}
- tone: {TONES[old_tone]}

You will be given the conversation history and the agent's strategic action.
Generate a realistic 1-2 sentence reply in the voice of someone using a dating app.
Then label the resulting state of the conversation along three dimensions and indicate any terminal outcome.

Outcome rules:
- outcome is null unless the conversation ended this turn.
- If the agent action is suggest_date, decide whether the date was accepted as "date_success" or rejected as "date_failed" based on the current state and conversation.
- If the other person would stop replying entirely, use "ghosted".
- You will not receive end_chat actions; those are handled outside the LLM.

Output ONLY valid JSON with exactly this schema:
{{
  "response": "<1-2 sentence reply>",
  "interest": "low" | "medium" | "high",
  "stage": "opener" | "chat" | "date",
  "tone": "cold" | "neutral" | "warm",
  "outcome": null | "date_success" | "date_failed" | "ghosted"
}}

No preamble. No explanation. No markdown."""

    user_prompt = f"""Conversation history:
{history_text}

Current state context:
{current_state_text}

Agent strategic action:
{action_name}

What this strategy means:
{action_description}

Respond as the other person, then classify the resulting conversation state."""

    response = client.chat.completions.create(
        model=KIMI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content
    data = json.loads(raw)
    expected_keys = {"response", "interest", "stage", "tone", "outcome"}
    if set(data) != expected_keys:
        raise ValueError(f"Invalid LLM world-model keys. Raw output: {raw!r}")

    if not isinstance(data["response"], str) or not data["response"].strip():
        raise ValueError(f"Invalid or empty response text. Raw output: {raw!r}")
    if data["interest"] not in INTEREST_MAP:
        raise ValueError(f"Invalid interest label. Raw output: {raw!r}")
    if data["stage"] not in STAGE_MAP:
        raise ValueError(f"Invalid stage label. Raw output: {raw!r}")
    if data["tone"] not in TONE_MAP:
        raise ValueError(f"Invalid tone label. Raw output: {raw!r}")
    if data["outcome"] not in LLM_OUTCOME_VALUES:
        raise ValueError(f"Invalid outcome label. Raw output: {raw!r}")

    new_state = (
        INTEREST_MAP[data["interest"]],
        STAGE_MAP[data["stage"]],
        TONE_MAP[data["tone"]],
    )
    next_state_id = state_to_id(*new_state)
    outcome = data["outcome"]
    done = outcome is not None
    reward = compute_reward(old_state, new_state, outcome)
    response_text = data["response"].strip()

    return next_state_id, reward, done, outcome, response_text


## 22. LLM World-Model Sanity Check

Run this once before attempting longer LLM-based rollouts. It makes one Kimi call, prints the generated reply, and shows the resulting state/reward/outcome.


In [ ]:
# LLM world-model sanity check.
# This spends one API call.
state = reset_episode()
action = ACTIONS.index("ask_question")
next_state, reward, done, outcome, response_text = simulator_step_llm(
    state_id=state,
    action=action,
    history=[],
)

print("Starting state:", describe_state(state))
print("Agent action:", ACTIONS[action])
print("LLM response:", response_text)
print("Next state:", describe_state(next_state))
print("Reward:", round(reward, 3))
print("Done:", done)
print("Outcome:", outcome)


## 23. run_episode with LLM Support


In [ ]:
def run_episode(
    Q=None,
    policy_fn=None,
    epsilon=0.0,
    max_steps=25,
    reset_fn=reset_episode,
    learn=False,
    alpha=0.1,
    gamma=0.95,
    use_llm=False,
):
    """Run one variable-length episode, optionally sourcing transitions from Kimi."""
    state_id = reset_fn()
    total_reward = 0.0

    if use_llm:
        history = []

    for step in range(max_steps):
        if policy_fn is not None:
            action = policy_fn(state_id)
        elif Q is None:
            action = np.random.randint(N_ACTIONS)
        else:
            action = choose_action(Q, state_id, epsilon)

        if use_llm:
            next_state_id, reward, done, outcome, response_text = simulator_step_llm(
                state_id,
                action,
                history,
            )
            history.append({"agent": ACTIONS[action], "other": response_text})
        else:
            next_state_id, reward, done, outcome = simulator_step(state_id, action)

        total_reward += reward

        if learn:
            best_next = np.max(Q[next_state_id])
            target = reward + (0.0 if done else gamma * best_next)
            Q[state_id, action] += alpha * (target - Q[state_id, action])

        state_id = next_state_id
        if done:
            return total_reward, step + 1, outcome

    return total_reward, max_steps, "max_steps"


## 24. run_episode_sarsa with LLM Support


In [ ]:
def run_episode_sarsa(
    Q,
    epsilon=0.0,
    max_steps=25,
    reset_fn=reset_episode,
    learn=False,
    alpha=0.1,
    gamma=0.95,
    use_llm=False,
):
    """Run one SARSA episode, optionally sourcing transitions from Kimi."""
    state_id = reset_fn()
    action = choose_action(Q, state_id, epsilon)
    total_reward = 0.0

    if use_llm:
        history = []

    for step in range(max_steps):
        if use_llm:
            next_state_id, reward, done, outcome, response_text = simulator_step_llm(
                state_id,
                action,
                history,
            )
            history.append({"agent": ACTIONS[action], "other": response_text})
        else:
            next_state_id, reward, done, outcome = simulator_step(state_id, action)

        total_reward += reward

        if done:
            if learn:
                Q[state_id, action] += alpha * (reward - Q[state_id, action])
            return total_reward, step + 1, outcome

        next_action = choose_action(Q, next_state_id, epsilon)

        if learn:
            target = reward + gamma * Q[next_state_id, next_action]
            Q[state_id, action] += alpha * (target - Q[state_id, action])

        state_id = next_state_id
        action = next_action

    return total_reward, max_steps, "max_steps"


## 25. LLM Training Wrappers


In [ ]:
def train_q_learning_llm(episodes=50, reset_fn=reset_episode, verbose=True, sleep_between=3.5):
    """Train tabular Q-learning using the LLM simulator. Prints progress.

    sleep_between: seconds to wait after each episode to respect Kimi's 20 RPM rate limit.
    With ~3-6 LLM calls per episode and 3.5s wait, we stay safely under 20 RPM.
    """
    Q_llm = np.zeros((N_STATES, N_ACTIONS))
    rewards = []

    for episode in range(episodes):
        epsilon = max(0.05, 1.0 - episode / (episodes * 0.75))
        try:
            reward, length, outcome = run_episode(
                Q=Q_llm,
                epsilon=epsilon,
                reset_fn=reset_fn,
                learn=True,
                alpha=0.1,
                gamma=0.95,
                use_llm=True,
            )
        except Exception as exc:
            print(f"Episode {episode + 1}/{episodes} failed: {exc}")
            time.sleep(sleep_between * 2)
            continue

        rewards.append(reward)
        if verbose:
            print(
                f"Episode {episode + 1}/{episodes}: "
                f"reward={reward:.3f}, length={length}, outcome={outcome}"
            )
        time.sleep(sleep_between)

    return Q_llm, rewards


def train_sarsa_llm(episodes=50, reset_fn=reset_episode, verbose=True, sleep_between=3.5):
    """Train tabular SARSA using the LLM simulator. Prints progress."""
    Q_llm = np.zeros((N_STATES, N_ACTIONS))
    rewards = []

    for episode in range(episodes):
        epsilon = max(0.05, 1.0 - episode / (episodes * 0.75))
        try:
            reward, length, outcome = run_episode_sarsa(
                Q=Q_llm,
                epsilon=epsilon,
                reset_fn=reset_fn,
                learn=True,
                alpha=0.1,
                gamma=0.95,
                use_llm=True,
            )
        except Exception as exc:
            print(f"Episode {episode + 1}/{episodes} failed: {exc}")
            time.sleep(sleep_between * 2)
            continue

        rewards.append(reward)
        if verbose:
            print(
                f"Episode {episode + 1}/{episodes}: "
                f"reward={reward:.3f}, length={length}, outcome={outcome}"
            )
        time.sleep(sleep_between)

    return Q_llm, rewards


## 26. Save/Load Q-Tables


In [ ]:
def save_q_table(Q_table, path):
    """Save a Q-table to disk so we don't have to re-train."""
    np.save(path, Q_table)
    print(f"Saved Q-table to {path}")


def load_q_table(path):
    """Load a saved Q-table."""
    Q_table = np.load(path)
    print(f"Loaded Q-table from {path}, shape {Q_table.shape}")
    return Q_table


## 27. 2-Episode Sanity Check (Spends ~$0.005)


In [ ]:
# Sanity check: 5 LLM episodes, with rate-limit-respecting sleeps.
# Cost ~$0.015. Will take ~30 seconds (5 episodes * 3.5s sleep + ~1s LLM calls).
print("Running 5 LLM-driven Q-learning episodes...")
Q_llm_test, rewards_test = train_q_learning_llm(episodes=5, verbose=True)
print(f"\nQ-table sum: {Q_llm_test.sum():.3f}")
print(f"Non-zero Q entries: {(Q_llm_test != 0).sum()} / {Q_llm_test.size}")
print(f"Outcomes: {[r for r in rewards_test]}")
